# 第五阶段：智能指针

## 实验 4：Custom Deleter —— 为非 `new` 资源定义销毁策略

`std::unique_ptr` 不只能够管理通过 `new` 创建的对象。只要提供 custom deleter，它也可以成为任意单一资源的 RAII owner。

本实验使用 C 标准库的 `FILE*`，因为它必须通过 `std::fclose()` 而不是 `delete` 释放。实验关注：

- custom deleter 如何成为智能指针类型的一部分；
- 正常返回、提前返回和异常路径是否都能关闭文件；
- 空智能指针为什么不会调用 deleter；
- 移动 `FilePtr` 时，文件 ownership 如何转移；
- 这种 C++ RAII owner 如何映射到 C ABI 和 Kotlin/Native。

本 Notebook 会自动创建相邻的 `../outputs/04/`，不再要求手工准备目录。

In [1]:
// 本步骤：引入文件、路径、智能指针、异常和断言所需的标准库。
#include <cassert>
#include <cstdio>
#include <filesystem>
#include <iostream>
#include <memory>
#include <stdexcept>
#include <string>
#include <type_traits>
#include <utility>

### 1. 把 `fclose` 封装成 destruction policy

`unique_ptr<T, Deleter>` 保存资源地址和 deleter。当智能指针拥有非空地址并需要释放时，它调用 `Deleter{}(pointer)`。因此：

```text
std::unique_ptr<std::FILE, FileCloser>
             │             │
             │ owns        └── destruction policy: fclose
             ▼
           FILE*
```

deleter 会从智能指针析构路径调用，必须保证不抛出 C++ 异常。

In [2]:
// 本步骤：定义 fclose 策略，并用计数器记录实际关闭次数。
struct FileCloser
{
    void operator()(std::FILE *file) const noexcept
    {
        // 防御性检查便于单独调用 deleter；unique_ptr 本身不会为空地址调用它。
        if (file == nullptr)
        {
            return;
        }

        // fclose 同时刷新缓冲区并释放 FILE 资源。
        std::fclose(file);
        ++close_count_;
        std::cout << "FILE closed, count = " << close_count_ << '\n';
    }

    static int close_count() noexcept
    {
        // 返回累计关闭次数，供不同控制流的断言使用。
        return close_count_;
    }

private:
    inline static int close_count_ = 0;
};

// 本步骤：为拥有 FILE* 的 unique_ptr 命名，让接口直接表达资源与销毁策略。
using FilePtr = std::unique_ptr<std::FILE, FileCloser>;

// deleter 会从析构路径调用，因此用编译期检查确保调用表达式为 noexcept。
static_assert(std::is_nothrow_invocable_v<FileCloser, std::FILE *>);

### 2. 工厂函数立即建立 ownership

`std::fopen()` 返回 owning raw pointer 或 `nullptr`。工厂函数立即把成功结果放进 `FilePtr`；失败时则返回一个空 `FilePtr`。这让调用方不需要接触裸 ownership。

In [3]:
// 本步骤：封装 fopen，把返回的 FILE* 立即交给 RAII owner。
FilePtr open_file(const std::filesystem::path &path, const char *mode)
{
    // path.string() 创建本地路径字符串，并在 fopen 调用期间保持有效。
    const std::string native_path = path.string();

    // FilePtr 接受 nullptr；成功时拥有文件，失败时保持空状态。
    return FilePtr(std::fopen(native_path.c_str(), mode));
}

### 3. 创建实验输出目录

初稿使用的路径在父目录不存在时会使 `fopen` 返回 `nullptr`。这里先显式创建目录，并输出最终绝对路径，方便确认 Jupyter kernel 的当前工作目录。

In [ ]:
// 本步骤：从 notebooks/ 定位相邻的 outputs/04，并保存后续示例共用的文件路径。
const std::filesystem::path output_directory =
    std::filesystem::current_path().parent_path() / "outputs" / "04";

// create_directories 会递归创建缺失目录；目录已存在时也不会报错。
std::filesystem::create_directories(output_directory);

// 所有示例操作同一文本文件，避免生成无关实验产物。
const std::filesystem::path output_path = output_directory / "test.txt";
std::cout << "output path: " << output_path << '\n';

### 4. 作用域结束时自动 `fclose`

写入时通过 `file.get()` 临时借用底层 `FILE*` 给 C 函数。`get()` 不转移 ownership，不能对返回值手动执行 `fclose()`；最终仍由 `FilePtr` 的 deleter 关闭文件。

In [5]:
// 本步骤：写入文件并观察 FilePtr 离开作用域时调用 custom deleter。
{
    const int before = FileCloser::close_count();

    {
        // open_file 返回 owner；若打开失败，FilePtr 为空。
        FilePtr file = open_file(output_path, "w");
        assert(file);

        // get() 只为 fputs 提供同步借用，file 继续拥有 FILE*。
        const int write_result = std::fputs("hello from custom deleter\n", file.get());
        assert(write_result >= 0);
    }

    // FilePtr 已析构，关闭计数应增加一次。
    assert(FileCloser::close_count() == before + 1);
}

FILE closed, count = 1


In [6]:
// 本步骤：重新打开文件读取内容，确认 fclose 已刷新先前的写缓冲区。
{
    FilePtr file = open_file(output_path, "r");
    assert(file);

    // fgets 借用 FILE* 并把一行内容写入本地字符数组。
    char buffer[128]{};
    const char *read_result = std::fgets(buffer, sizeof(buffer), file.get());
    assert(read_result != nullptr);
    assert(std::string(buffer) == "hello from custom deleter\n");
    std::cout << "read back: " << buffer;

    // 当前作用域结束时，读取用的 FILE* 也会自动关闭。
}

read back: hello from custom deleter
FILE closed, count = 2


### 5. 打开失败得到空 owner

`unique_ptr` 只在持有非空 pointer 时调用 deleter。因此 `fopen` 失败不会触发 `fclose(nullptr)`。失败分支应在普通函数中返回状态或错误类型，而不是从 Notebook 顶层执行 `return`。

In [7]:
// 本步骤：用空路径稳定触发 fopen 失败，并确认 deleter 没有被调用。
{
    const int before = FileCloser::close_count();

    // fopen 失败返回 nullptr，FilePtr 因而处于空状态。
    FilePtr missing = open_file("", "r");
    assert(!missing);

    // 没有获得资源，就不应发生 fclose 调用。
    missing.reset();
    assert(FileCloser::close_count() == before);
    std::cout << "open failure represented by empty FilePtr\n";
}

open failure represented by empty FilePtr


### 6. 提前返回仍会关闭文件

与普通 `unique_ptr<User>` 一样，custom deleter 不依赖函数走到最后一行。局部 `FilePtr` 在任意正常返回路径上都会析构。

In [8]:
// 本步骤：定义带提前返回分支的写入函数，返回 bool 表示打开是否成功。
bool append_message(const std::filesystem::path &path, bool early_return)
{
    // FilePtr 在函数作用域内拥有追加模式打开的文件。
    FilePtr file = open_file(path, "a");
    if (!file)
    {
        // 没有取得资源时直接报告失败，不执行文件操作。
        return false;
    }

    // 向 C API 临时借出 FILE*；ownership 仍属于 file。
    std::fputs("append before return\n", file.get());

    if (early_return)
    {
        // return 前会析构 file，因此 custom deleter 仍会执行。
        return true;
    }

    // 正常路径也无需手写 fclose。
    std::fputs("append on normal path\n", file.get());
    return true;
}

In [9]:
// 本步骤：分别执行提前返回和正常返回，验证两条路径各关闭一次文件。
{
    const int before = FileCloser::close_count();

    // 两次调用都成功取得文件，并在各自返回时自动关闭。
    assert(append_message(output_path, true));
    assert(append_message(output_path, false));

    // 每个局部 FilePtr 都恰好调用一次 deleter。
    assert(FileCloser::close_count() == before + 2);
}

FILE closed, count = 3
FILE closed, count = 4


### 7. 移动转移资源和 deleter

`FilePtr` 仍然是 `unique_ptr`：不可复制，但可以移动。移动后源 owner 为空，目标 owner 保存同一个 `FILE*`，最终只调用一次 `fclose`。

In [10]:
// 本步骤：移动 FilePtr，确认底层 FILE* 身份不变且只关闭一次。
{
    const int before = FileCloser::close_count();

    {
        FilePtr source = open_file(output_path, "a");
        assert(source);

        // 记录非拥有地址，仅用于验证移动前后是同一个文件资源。
        std::FILE *address = source.get();

        // 移动构造把 FILE* 及其 deleter policy 一起交给 destination。
        FilePtr destination = std::move(source);
        assert(!source);
        assert(destination.get() == address);

        // 目标 owner 继续把 FILE* 借给 C 写入函数。
        std::fputs("append after move\n", destination.get());
    }

    // 只有 destination 析构时执行一次 fclose。
    assert(FileCloser::close_count() == before + 1);
}

FILE closed, count = 5


### 8. 异常路径同样由 RAII 清理

当异常离开函数时，已经完成构造的局部对象会依次析构。只要 `FilePtr` 已经接管 `FILE*`，后续 C++ 操作抛出异常也不会泄漏文件。异常不能跨 C ABI；边界层仍应捕获并转换为状态码。

In [11]:
// 本步骤：打开文件后故意抛出异常，观察栈展开调用 custom deleter。
void write_then_throw(const std::filesystem::path &path)
{
    // 成功打开后，file 立即成为 FILE* 的 owner。
    FilePtr file = open_file(path, "a");
    if (!file)
    {
        // 用 C++ 异常报告无法继续的内部错误。
        throw std::runtime_error("failed to open output file");
    }

    // 先写入可观察内容，再模拟后续业务逻辑失败。
    std::fputs("append before exception\n", file.get());
    throw std::runtime_error("simulated processing error");
}

In [12]:
// 本步骤：捕获模拟异常，并确认栈展开期间文件已经关闭。
{
    const int before = FileCloser::close_count();

    try
    {
        // 函数会在 FilePtr 建立 ownership 后抛出异常。
        write_then_throw(output_path);
    }
    catch (const std::runtime_error &error)
    {
        // 异常到达这里之前，局部 FilePtr 已执行析构和 fclose。
        std::cout << "caught: " << error.what() << '\n';
    }

    assert(FileCloser::close_count() == before + 1);
}

FILE closed, count = 6
caught: simulated processing error


### 9. Custom deleter 是类型和 API contract 的一部分

这些 owner 看起来都保存某种地址，但销毁策略完全不同：

```cpp
std::unique_ptr<User>                 // delete User
std::unique_ptr<std::FILE, FileCloser> // fclose(FILE*)
```

`FilePtr` 不能随意转换成使用另一种 deleter 的 `unique_ptr`。类型系统借此阻止 owner 在转移过程中忘记正确的资源释放协议。相同模式还可用于 socket 的 `close`、SQLite 的 `sqlite3_close`，以及 SDK 自己的 `sdk_destroy`。

需要注意：`release()` 会取走 pointer 并取消自动调用 deleter。除非另一个 API 明确接管资源，否则不要对 `FilePtr` 使用 `release()`。

### 10. C++ Core、C ABI 与 Kotlin/Native

`FilePtr` 很适合作为 C++ Core 的私有实现类型，但不能直接出现在 C ABI：它涉及 C++ 模板、对象布局、析构函数和标准库 ABI。跨语言边界应只暴露 C 兼容 handle 和成对生命周期函数：

```text
C++ Core
FilePtr / custom deleter
        │
════════ C ABI Boundary ════════
        │
sdk_file_handle* sdk_file_open(...)
void sdk_file_close(sdk_file_handle*)
        │
Kotlin/Native owner / close()
```

C++ 实现可在 handle 内部继续用 RAII；C ABI 的 `close/destroy` 负责触发 owner 析构。Kotlin 包装层必须明确谁调用关闭函数，不能只等待普通 Kotlin 引用离开作用域。

## 实验结论

`std::unique_ptr<T, Deleter>` 可以读成：**唯一拥有一个 `pointer`，并在 ownership 结束时执行指定的 destruction policy。**

需要记住：

1. `FILE*` 由 `fopen` 创建，必须由 `fclose` 释放，不能使用 `delete`；
2. `FilePtr` 把 pointer、唯一 ownership 和 `FileCloser` 绑定在同一个 RAII 类型中；
3. `get()` 只提供同步借用，不改变 ownership；
4. 正常返回、提前返回和异常栈展开都会析构 `FilePtr`；
5. 空 `FilePtr` 没有资源，因此不会调用 deleter；
6. custom deleter 应为 `noexcept`，不能让异常从析构路径逃逸；
7. 跨 C ABI 时把内部 RAII owner 转换为 handle 与显式 `destroy/close`。